# Bổ trợ NLP (60 phút)
    Từ câu tiếng Việt đến bảng số và trạng thái chuỗi. Ví dụ dùng dữ liệu tự viết, không phải test.
    Tra cứu `../appendix/C_NLP.md`. Sau bài này bạn tự xây vocabulary và LSTM trong project.

In [ ]:
import os, sys
from pathlib import Path
# Chạy tại root repo hoặc thư mục con; tìm root có shared/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'shared/data.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Giải nén/clone toàn bộ repo, rồi chuyển thư mục tới nơi có shared/. Xem CLOUD.md.')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import torch
import matplotlib.pyplot as plt
from shared.data import load_data, describe, normalized_images
from shared.baselines import run_baselines
from shared.utils import load_config, read_json
from shared.engine import fit, make_loader, evaluate, reload_model, predict_image, predict_text
from shared.metrics import plot_confusion, plot_history, show_errors
from shared.protocol import register_protocol, lock_selection, final_test
from shared.kiem_tra import check_model, check_padding
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(4)
DATA_ROOT = Path(os.environ.get('AI4B_DATA_ROOT', 'data'))
OUTPUT_ROOT = Path(os.environ.get('AI4B_OUTPUT_ROOT', 'runs/my_project'))
SMOKE = os.environ.get('AI4B_SMOKE') == '1'  # Chỉ kiểm kỹ thuật, không dùng để báo cáo benchmark.
if SMOKE: OUTPUT_ROOT = OUTPUT_ROOT / 'smoke'
print('PyTorch', torch.__version__, '| device:', DEVICE)
print('Data:', DATA_ROOT.resolve(), '| Output:', OUTPUT_ROOT.resolve(), '| smoke:', SMOKE)

## 1. Token, PAD và UNK (0–12 phút)
    Chuẩn hóa NFC và viết thường, tách theo khoảng trắng: đây là token âm tiết, chưa phải word segmentation.
    Vocabulary ví dụ cố định bên dưới chỉ minh họa; vocabulary project phải học từ train.
    Quan sát từ chưa biết thành UNK, PAD chỉ để ghép batch. Vì sao PAD và UNK phải khác ID?

In [ ]:
from shared.data import normalize
text = 'Giảng viên nhiệt tình'
toy_vocab = {'<PAD>':0, '<UNK>':1, 'giảng':2, 'viên':3, 'nhiệt':4, 'tình':5}
tokens = normalize(text).split()
print(tokens, [toy_vocab.get(t,1) for t in tokens])
print('từ mới:', toy_vocab.get('zzzz', 1))

## 2. Embedding là bảng tra (12–23 phút)
    Hàng thứ i của E là vector token i. One-hot nhân E cho cùng vector.
    Làm: chạy ô, xác nhận hai kết quả giống nhau. Giải thích embedding khác label encoding ở đâu.

In [ ]:
E = torch.arange(18, dtype=torch.float32).reshape(6,3) / 10
ids = torch.tensor([2,3,4,5])
one_hot = torch.nn.functional.one_hot(ids, num_classes=6).float()
print(E[ids]); print(one_hot @ E)
assert torch.equal(E[ids], one_hot @ E)

## 3. Một bước RNN bằng số nhỏ (23–35 phút)
    `h_t = tanh(x_t W_x + h_(t-1) W_h + b)`.
    Tính tay bước đầu, so kết quả. Đảo thứ tự token và quan sát hidden cuối.
    Vì sao TF-IDF và RNN mô tả thứ tự khác nhau? TF-IDF bigram vẫn có thông tin cục bộ.

In [ ]:
Wx = torch.tensor([[.2,.1],[-.1,.3],[.4,-.2]])
Wh = torch.tensor([[.5,0.],[0.,.5]])
b = torch.zeros(2)
def recurrent_trace(sequence):
    h = torch.zeros(2)
    states = []
    for token in sequence:
        h = torch.tanh(E[token] @ Wx + h @ Wh + b)
        states.append(h.clone())
    return torch.stack(states)
print(recurrent_trace(ids)); print(recurrent_trace(ids.flip(0)))

## 4. LSTM giữ và cập nhật trạng thái (35–46 phút)
    RNN lặp nhiều lần dễ mất hoặc bùng gradient. LSTM thêm cell state và các cổng.
    Forget gate giữ phần cũ; input gate chọn nội dung mới; output gate điều tiết hidden.
    Đọc công thức trong Appendix C. `h_n` khác `c_n` thế nào?
    Many-to-one: một hidden cuối → một nhãn cho cả câu, không phải một nhãn/token.

## 5. Padding không phải từ thật (46–54 phút)
    Hai câu dài 2 và 5 token cần ghép thành batch. Nếu dùng output tại PAD cuối, trạng thái có thể thay đổi.
    Làm: viết tensor ID có PAD bằng tay và tensor length tương ứng; length sau cắt phải ≤40 và >0.
    Project sẽ cung cấp khung `pack_padded_sequence`; bạn hoàn thiện embedding, LSTM và head.
    Kiểm cùng câu chạy riêng và trong batch có PAD là điều kiện bắt buộc.

In [ ]:
toy_ids = torch.tensor([[2,3,0,0,0], [2,3,4,5,1]], dtype=torch.long)
lengths = torch.tensor([2,5], dtype=torch.long)
print(toy_ids, lengths)
assert torch.equal((toy_ids != 0).sum(1), lengths)

## 6. Gradient clipping (54–60 phút)
    Nếu norm gradient là 10 và max norm=1, vector giữ hướng và giảm độ dài 10 lần.
    Clipping thực hiện sau backward, trước step; khác điều chỉnh learning rate hoặc class weighting.
    Tự kiểm: giải thích UNK, length, `h_n`, neutral và macro-F1 trong 60 giây.
    Tiếp: mở project NLP, chạy TF-IDF và điền TODO N1–N4.